In [1]:
import pandas as pd 
import numpy as np 
import os 
import glob
from IPython.display import display


In [2]:
raw_data_dir = "../01_Raw_Data/01_raw_signals"
processed_data_dir = "../02_Processed_Data/01_Sep_Cutting_Lines"

#create process data
os.makedirs(processed_data_dir, exist_ok=True)

#find all csv files in the specified folder 
file_paths = glob.glob(os.path.join(raw_data_dir, "*.csv"))
summary_data = []


trimmed_file_count = 0
successful_file_count=0

#function to find columns
def find_column(df, keyword):
    for col in df.columns:
        if str(col).startswith(keyword):
            return col
    return None

for file_path in file_paths:
    file_name = os.path.basename(file_path)

    #*********************************
    # Skip meta data
    #*********************************
    header_idx = None
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            for i, line in enumerate(f):
                if 'Z_Kafa_HMI' in line or 'Kafa.Act.Pos' in line:
                    header_idx = i
                    break
    except Exception as e:
        summary_data.append({'File Name': file_name, 'Status': f'Read error: {e}'})
        continue

    if header_idx is None:
        summary_data.append({'File Name': file_name, 'Statues': 'Invalid Record: Columns headers not found!'})
        continue

    try:
        df = pd.read_csv(file_path, skiprows=header_idx, sep=None, engine='python')
    except Exception as e:
        summary_data.append({'File Name': file_name, 'Status': f'CSV Parse Error: {e}'})
        continue
        

    #************************************
    # Find Columns and Convert to Numeric
    # ***********************************
    col_vel = find_column(df, "Kafa.Act.Vel")
    col_pos = find_column(df, "Kafa.Act.Pos")
    col_z_hmi = find_column(df, "Z_Kafa_HMI")
    col_trq = find_column(df, "Kafa.Act.Trq") 
    
    if not all([col_vel, col_pos, col_z_hmi, col_trq]):
        summary_data.append({'Dosya Adı': file_name, 'Status': 'Invalid Record: Z_HMI, Pos, Vel or Trq columns is missing'})
        continue

    # replace commas with dots and convert to numeric (file D has commas and dots)
    for c in [col_vel, col_pos, col_z_hmi, col_trq]:
        if c in df.columns:
            df[c] = df[c].astype(str).str.replace(',', '.').apply(pd.to_numeric, errors='coerce')   

    #*************************************************
    #Cutting Kinematics (Finding starts an end points)
    #*************************************************
    steady_condition = (df[col_z_hmi] < 0) & (df[col_pos] < 444)
    
    if steady_condition.any():
        start_idx = df[steady_condition].index.min()
    else:
        summary_data.append({'File Name': file_name, 'Status': "Invalid Record: 444 and Z<0 contidion never met"})
        continue

    end_condition = df.loc[start_idx:][col_vel] >= -0.5
    
    if end_condition.any():
        end_idx = df.loc[start_idx:][end_condition].index.min() - 1
    else:
        end_idx = df.index.max()

    if pd.isna(end_idx): 
        end_idx = df.index.max()

    # Isolate only the cutting region
    df_cutting = df.loc[start_idx:end_idx].copy().reset_index(drop=True)

    #**************************************
    #Tork Control and Conversion to Positive
    #***************************************
    #Find the index of the LAST row where the torque is negative (or at most 0)
    #last_negative_torque_idx = df_cutting[df_cutting[col_trq] <= 0].index.max()
    
    #if pd.notna(son_gecerli_index):
        #Discard the positive torque tail
        #df_cutting = df_cutting.loc[:last_negative_torque_idx].copy()
        # trimmed_file_count+= 1
        
    #Convert all values in the torque column to positive using absolute value
    #df_cutting[col_trq] = df_cutting[col_trq].abs()


    #****************************
    #SAve celaned data as csv
    #****************************
    saved_filename = f'cutting_{file_name}'
    processed_file_path = os.path.join(processed_data_dir, saved_filename)
    df_cutting.to_csv(processed_file_path, index=False)
    
    successful_file_count += 1
    
    # Log tablosu için kayıt ekle
    excel_start = int(start_idx) + header_idx + 2
    excel_end = int(end_idx) + header_idx + 2
    summary_data.append({
        'File Name': file_name, 
        'Status': 'Success',
        'Excel Start Index': excel_start, 
        'Excel End Index': excel_end
    })

#Display thr results as a table
summary_df = pd.DataFrame(summary_data)
display(summary_df)


,File Name,Status,Excel Start Index,Excel End Index
0,6.csv,Success,89,1292
1,7.csv,Success,90,1207
2,A.csv,Success,106,1528
3,C.csv,Success,89,1341
4,5.csv,Success,104,1407
5,4.csv,Success,127,1346
6,B.csv,Success,81,1446
7,F.csv,Success,100,1413
8,G.csv,Success,102,1305
9,1.csv,Success,92,1799
